# ============================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================

In [ ]:
import sys
print(sys.executable)

import pandas as pd
print("Pandas funcionando!")
print(pd.__version__)

RANDOM_STATE = 42

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)

from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# ============================================
# 2. CARREGAMENTO DAS BASES
# ============================================

In [ ]:
import pandas as pd
from pathlib import Path

# Pasta onde estão os arquivos
pasta_dados = Path("../data/raw")

# Caminho dos arquivos
arquivo_application = pasta_dados / "application_record.csv"
arquivo_credit = pasta_dados / "credit_record.csv"

# Verificar se os arquivos existem
print("Application existe?", arquivo_application.exists())
print("Credit existe?", arquivo_credit.exists())

# Carregar as bases
application = pd.read_csv(arquivo_application)
credit = pd.read_csv(arquivo_credit)

# Verificar o tamanho das bases
print("\nBase APPLICATION:")
print(application.shape)

print("\nBase CREDIT:")
print(credit.shape)

# ============================================
# 3. ANÁLISE E TRATAMENTO DOS DADOS
# ============================================

In [ ]:
# Informações da base de aplicações
display(application.head())

In [ ]:
# Informações da base de histórico de crédito
display(credit.head())

In [ ]:
print("Colunas da APPLICATION:")
print(application.columns.tolist())

print("\nColunas da CREDIT:")
print(credit.columns.tolist())

In [ ]:
application = pd.read_csv(arquivo_application)
credit = pd.read_csv(arquivo_credit)

In [ ]:
print("Application Record:")
print(application.shape)

print("\nCredit Record:")
print(credit.shape)

Essa é a base que contém as características do cliente.

In [ ]:
application.head()

Um mesmo cliente aparece várias vezes, ou seja, uma linha = um mês de histórico de crédito.

In [ ]:
credit.head(20)

In [ ]:
application.info()
credit.info()
application.describe(include="all").T

**Verificar valores ausentes**

In [ ]:
application.isnull().sum().sort_values(ascending=False)
missing = (
    application.isnull()
    .mean()
    .sort_values(ascending=False) * 100
)

missing[missing > 0]

**Verificar duplicidades**

In [ ]:
application["ID"].duplicated().sum()
application[application["ID"].duplicated(keep=False)].head(20)

In [200]:
# Verificar cadastros idênticos com IDs diferentes

print("Colunas disponíveis:")
print(application.columns.tolist())

# Separar as colunas cadastrais, excluindo o ID
colunas_cadastro = [
    coluna for coluna in application.columns
    if coluna != "ID"
]

# Identificar registros iguais em todas as colunas cadastrais
duplicados_cadastro = application[
    application.duplicated(
        subset=colunas_cadastro,
        keep=False
    )
].sort_values(by=colunas_cadastro)

print(
    "\nQuantidade de registros envolvidos em duplicidade cadastral:",
    len(duplicados_cadastro)
)

if not duplicados_cadastro.empty:
    quantidade_grupos = duplicados_cadastro[
        colunas_cadastro
    ].drop_duplicates().shape[0]

    print(
        "Quantidade de cadastros distintos com duplicidade:",
        quantidade_grupos
    )

    print("\nAmostra de até 10 registros encontrados:")
    print(
        duplicados_cadastro[
            ["ID"] + colunas_cadastro
        ].head(10).to_string(index=False)
    )

    print("\nAmostra de até 10 cadastros com seus números de IDs:")
    ids_por_cadastro = (
        duplicados_cadastro.groupby(
            colunas_cadastro,
            dropna=False
        )["ID"]
        .nunique()
        .reset_index(name="quantidade_ids")
    )

    print(ids_por_cadastro.head(10).to_string(index=False))

    print(
        "\nA análise completa foi mantida em duplicados_cadastro. "
        "Apenas a exibição foi limitada a 10 registros e 10 grupos."
    )
else:
    print(
        "Nenhum cadastro idêntico foi encontrado "
        "considerando todas as colunas, exceto ID."
    )

Colunas disponíveis:
['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS']

Quantidade de registros envolvidos em duplicidade cadastral: 423205
Quantidade de cadastros distintos com duplicidade: 74780

Amostra de até 10 registros encontrados:
     ID CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL NAME_INCOME_TYPE           NAME_EDUCATION_TYPE NAME_FAMILY_STATUS NAME_HOUSING_TYPE  DAYS_BIRTH  DAYS_EMPLOYED  FLAG_MOBIL  FLAG_WORK_PHONE  FLAG_PHONE  FLAG_EMAIL OCCUPATION_TYPE  CNT_FAM_MEMBERS
6093713           F            N               N             0           26100.0        Pensioner Secondary / secondary special            Married House / apartment      -21003         365243           1                0    

In [ ]:
# Confirmar cadastros idênticos associados a IDs diferentes

colunas_cadastro = [
    coluna for coluna in application.columns
    if coluna != "ID"
]

duplicados_cadastro = application[
    application.duplicated(
        subset=colunas_cadastro,
        keep=False
    )
]

grupos = duplicados_cadastro.groupby(
    colunas_cadastro,
    dropna=False
)["ID"].agg(["nunique", "size"])

# Selecionar apenas cadastros associados a mais de um ID
grupos_ids_diferentes = grupos[
    grupos["nunique"] > 1
]

print(
    "Quantidade de cadastros idênticos com IDs diferentes:",
    len(grupos_ids_diferentes)
)

print(
    "Total de registros nesses grupos:",
    grupos_ids_diferentes["size"].sum()
)

print("\nExemplos de grupos encontrados:")
print(grupos_ids_diferentes.head(10))

Para o nosso modelo, precisamos ter uma linha por cliente.

In [ ]:
application = application.drop_duplicates(subset="ID", keep="first")
print(application.shape)
print(application["ID"].nunique())

**Análise do histórico de crédito**

In [ ]:
credit["STATUS"].value_counts()

In [ ]:
credit["STATUS"].value_counts(normalize=True) * 100

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=credit,
    x="STATUS",
    order=sorted(credit["STATUS"].unique())
)

plt.title("Distribuição dos Status de Crédito")
plt.xlabel("Status")
plt.ylabel("Quantidade")
plt.show()

O gráfico apresenta a distribuição dos registros de acordo com o status de crédito. Observa-se uma concentração significativa nos status **C** e **0**, enquanto os demais status apresentam frequências consideravelmente menores. Os status **C** e **0** representam a maior parte das observações da base, indicando que determinados comportamentos de pagamento são predominantes no histórico analisado.

Essa distribuição evidencia a necessidade de compreender o significado de cada categoria antes da construção da variável-alvo. Como o objetivo do projeto é classificar os clientes em **bons e maus pagadores**, os diferentes status deverão ser agrupados de acordo com sua relação com o comportamento de pagamento e com a ocorrência de inadimplência.

Portanto, esse gráfico é de distribuição dos status históricos, e não ainda de bons versus maus pagadores

**Construção de variável alvo**

Status cosiderado como inadimplência

In [ ]:
bad_status=["1", "2", "3", "4", "5"]

In [ ]:
credit["BAD_STATUS"] = credit["STATUS"].isin(bad_status).astype(int)

In [ ]:
credit.head()

**HIstórico mensal em classificação do cliente.**

Se o cliente teve pelo menos um status ruim, ele será classificado como mau pagador.

In [ ]:
target = (
    credit
    .groupby("ID")["BAD_STATUS"]
    .max()
    .reset_index()
)

In [ ]:
target = target.rename(columns={
    "BAD_STATUS": "TARGET"
})

TARGET = 0 → bom pagador

TARGET = 1 → mau pagador

In [ ]:
target.head()

distribuição da variável-alvo

In [ ]:
target["TARGET"].value_counts()

In [ ]:
target["TARGET"].value_counts(normalize=True) * 100

Isso revela um problema importante:
Desbalanceamento de classes.
Temos muito mais bons pagadores do que maus pagadores.
Por isso accuracy sozinha não é uma boa métrica.
Desse modo vamos usar:
Precision, Recall, F1, ROC-AUC, PR-AUC,  e Balanced Accuracy

Juntar as duas bases.
Esse será nosso dataset final de modelagem.

In [ ]:
df = application.merge(
    target,
    on="ID",
    how="inner"
)

In [ ]:
print("Linhas em application:", len(application))
print("IDs únicos em application:", application["ID"].nunique())

print("\nLinhas em target:", len(target))
print("IDs únicos em target:", target["ID"].nunique())

print("\nLinhas em df após o merge:", len(df))
print("IDs únicos em df:", df["ID"].nunique())

print("\nIDs duplicados em application:",
      application["ID"].duplicated().sum())

print("IDs duplicados em target:",
      target["ID"].duplicated().sum())

print("IDs duplicados em df:",
      df["ID"].duplicated().sum())

Esses dados confirmam que não há duplicação de IDs nas tabelas que estamos usando

In [ ]:
ids_application = set(application["ID"])
ids_target = set(target["ID"])
ids_df = set(df["ID"])

print("IDs de application que não estão em target:",
      len(ids_application - ids_target))

print("IDs de target que não estão em application:",
      len(ids_target - ids_application))

print("IDs de target que entraram em df:",
      len(ids_df))

print("Total de IDs em target:",
      len(ids_target))

print("Percentual de target mantido no df:",
      round(len(ids_df) / len(ids_target) * 100, 2), "%")

9.528 clientes da tabela target não têm correspondência em application e ficaram de fora.

In [ ]:
print("Distribuição de TARGET em target:")
print(target["TARGET"].value_counts())
print("\nProporção em target:")
print(target["TARGET"].value_counts(normalize=True).round(3))

print("\nDistribuição de TARGET em df:")
print(df["TARGET"].value_counts())
print("\nProporção em df:")
print(df["TARGET"].value_counts(normalize=True).round(3))

a proporção de bons e maus pagadores praticamente não mudou após o merge. Isso indica que a junção não alterou significativamente o equilíbrio das classes, embora ainda existam clientes sem correspondência entre as bases.

In [ ]:
print("Quantidade de IDs distintos em credit:",
      credit["ID"].nunique())

print("Quantidade de IDs distintos em target:",
      target["ID"].nunique())

print("Quantidade de IDs distintos em application:",
      application["ID"].nunique())

print("\nPrimeiros IDs de credit:")
print(credit["ID"].head().tolist())

print("\nPrimeiros IDs de application:")
print(application["ID"].head().tolist())

In [ ]:
print(df.shape)
df.head()

**Verificação do target**

In [ ]:
sns.countplot(data=df, x="TARGET")

plt.title("Distribuição dos Bons e Maus Pagadores")
plt.xlabel("0 = Bom | 1 = Mau")
plt.ylabel("Quantidade")
plt.show()

O gráfico apresenta a distribuição dos clientes entre as duas classes utilizadas no modelo de classificação: **bons pagadores (0)** e **maus pagadores (1)**.

Observa-se uma predominância significativa de bons pagadores, com aproximadamente **32 mil registros**, enquanto a quantidade de maus pagadores é inferior a **5 mil registros**. Dessa forma, verifica-se um **desbalanceamento entre as classes**, com a classe de bons pagadores representando a maioria das observações.

**Análise exploratória — idade**

In [ ]:
df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25

Na base a idade está números negativos. Vamos transformar em anos.

In [ ]:
df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25

In [ ]:
df["AGE_YEARS"].describe()

Agora é possível comparar bons e maus pagadores por idade

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    data=df,
    x="AGE_YEARS",
    hue="TARGET",
    bins=30,
    kde=True,
    stat="density",
    common_norm=False
)

plt.title("Distribuição de Idade por Classificação")
plt.xlabel("Idade")
plt.ylabel("Densidade")
plt.show()

A variável Idade demonstra uma capacidade de diferenciação clara entre as duas classes: clientes/utilizadores mais jovens têm uma tendência proporcionalmente maior para pertencer à classe Target 1 (maus pagadores), enquanto perfis de meia-idade estão mais associados à classe TARGET 0 (bons pagadores). Recomendam-se estratégias segmentadas por faixa etária com base no perfil de risco ou comportamento associado a cada target(alvo).  

**Análise da renda**

In [ ]:
df["AMT_INCOME_TOTAL"].describe()

Podemos observar que renda costuma ter distribuição bastante assimétrica.

In [ ]:
df["INCOME_LOG"] = np.log1p(df["AMT_INCOME_TOTAL"])

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    data=df,
    x="INCOME_LOG",
    hue="TARGET",
    bins=40,
    kde=True,
    stat="density",
    common_norm=False
)

plt.title("Distribuição da Renda Transformada")
plt.xlabel("log(1 + renda)")
plt.ylabel("Densidade")
plt.show()

O gráfico apresenta a distribuição da renda transformada em escala logarítmica para dois grupos de clientes, TARGET 0 (bons pagadores) e TARGET 1 (maus pagadores). A utilização do logaritmo serve para equilibrar os dados, reduzindo o efeito de valores muito altos ou muito baixos.   
 
A principal conclusão é que as curvas dos dois grupos são praticamente idênticas, concentrando-se quase totalmente no mesmo intervalo (entre 11,5 e 12,5). Isto significa que a renda, por si só, não permite distinguir se um cliente pertence ao grupo 0 ou ao grupo 1.   
 
Para que este dado tenha utilidade prática na tomada de decisão ou em modelos de avaliação de risco, é indispensável combiná-lo com outras métricas, como o histórico financeiro ou a capacidade de pagamento do cliente.   

**Renda por número de pessoas da família**

In [ ]:
df["INCOME_PER_FAMILY"] = (
    df["AMT_INCOME_TOTAL"] /
    df["CNT_FAM_MEMBERS"].replace(0, np.nan)
)

**Tempo de emprego**

Essa variável tem dois problemas:

1- está em dias negativos.

2- Existe um valor que se repete "365243", que representa uma codificação de ausência de informação.

In [ ]:
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(
    365243,
    np.nan
)

In [ ]:
df["EMPLOYED_YEARS"] = -df["DAYS_EMPLOYED"] / 365.25

**Testando algumas relações categóricas**

tipo de renda

como TARGET vale 0 ou 1, a média representa a proporção de maus pagadores.

In [ ]:
income_type_target = (
    df.groupby("NAME_INCOME_TYPE")["TARGET"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)

income_type_target

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    data=df,
    x="NAME_INCOME_TYPE",
    y="TARGET"
)

plt.xticks(rotation=45, ha="right")
plt.title("Taxa de Maus Pagadores por Tipo de Renda")
plt.xlabel("Tipo de renda")
plt.ylabel("Proporção de maus pagadores")
plt.show()

O gráfico analisa a taxa de maus pagadores por tipo de renda, destacando a proporção média de inadimplência e a respetiva margem de erro para cada categoria. As quatro principais categorias profissionais — trabalhadores, sócios comerciais, aposentados e funcionários públicos  — apresentam taxas de inadimplência muito consistentes e próximas entre si, variando apenas entre 10% e 13%. Em contrapartida, o segmento dos estudantes regista uma taxa média em torno de 9%, mas apresenta uma margem de erro extremamente ampla devido ao reduzido tamanho da amostra, o que torna o seu resultado estatisticamente incerto. Em suma, o tipo de renda isoladamente oferece reduzido poder para diferenciar o risco de crédito entre as grandes classes de clientes, recomendando-se a sua combinação com métricas financeiras mais robustas.

*Obs.: isso mostra associação nos dados, não significa que o tipo de renda seja a causa da inadimplência.*

**Escolaridade**

In [ ]:
education_target = (
    df.groupby("NAME_EDUCATION_TYPE")["TARGET"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)

education_target

# ============================================
# MODELAGEM
# ============================================

Não queremos usar:

ID

TARGET

DAYS_BIRTH

DAYS_EMPLOYED

AMT_INCOME_TOTAL

porque:

ID é identificador;

TARGET é justamente o que queremos prever;

criamos versões mais interpretáveis de idade e emprego;

criamos INCOME_LOG para representar renda de forma mais adequada.

In [ ]:
X = df.drop(
    columns=[
        "ID",
        "TARGET",
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "AMT_INCOME_TOTAL"
    ]
)

y = df["TARGET"]

In [ ]:
print(X.shape)
print(y.shape)

**Separando variáveis numéricas e categóricas**

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Variáveis numéricas:")
print(numeric_features)

print("\nVariáveis categóricas:")
print(categorical_features)

**Separar treino e teste**

Decisão do grupo:

80% treinamento

20% teste

*Obs.: Isso irá preservar a mesma proporção de bons e maus pagadores nos dois conjuntos.*

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Conferir se cada ID identifica um único cadastro na base original
if not application["ID"].is_unique:
    raise ValueError(
        "A base application contém IDs repetidos. "
        "Precisamos investigar esses IDs antes de continuar."
    )

# Criar um grupo para cada combinação de dados cadastrais,
# desconsiderando a coluna ID
colunas_cadastro = [
    coluna for coluna in application.columns
    if coluna != "ID"
]

application_com_grupo = application.copy()

application_com_grupo["grupo_duplicidade"] = (
    application_com_grupo
    .groupby(colunas_cadastro, dropna=False)
    .ngroup()
)

# Associar cada ID ao respectivo grupo cadastral
grupo_por_id = application_com_grupo.set_index("ID")[
    "grupo_duplicidade"
]

# Alinhar os grupos com as linhas utilizadas na modelagem
grupos = df["ID"].map(grupo_por_id)

if grupos.isna().any():
    raise ValueError(
        "Existem IDs de df que não foram encontrados em application."
    )

print("Total de linhas na modelagem:", len(df))
print("Total de grupos cadastrais:", grupos.nunique())
print("Grupos com mais de um registro:",
      grupos.value_counts().gt(1).sum())

Na base efetivamente usada pelo modelo, existem muitos registros agrupados por características cadastrais idênticas.

In [ ]:
# Divisão estratificada e agrupada: aproximadamente 80% / 20%

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    cv.split(X, y, groups=grupos)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Confirmar que nenhum grupo aparece nos dois conjuntos
grupos_treino = set(grupos.iloc[train_idx])
grupos_teste = set(grupos.iloc[test_idx])

sobreposicao = grupos_treino.intersection(grupos_teste)

print("Registros de treino:", len(X_train))
print("Registros de teste:", len(X_test))
print("Proporção de teste:",
      round(len(X_test) / len(X), 3))
print("Grupos compartilhados entre treino e teste:",
      len(sobreposicao))

print("\nProporção de TARGET no conjunto completo:")
print(y.value_counts(normalize=True).round(3))

print("\nProporção de TARGET no treino:")
print(y_train.value_counts(normalize=True).round(3))

print("\nProporção de TARGET no teste:")
print(y_test.value_counts(normalize=True).round(3))

assert len(sobreposicao) == 0, (
    "Erro: há grupos cadastrais compartilhados entre treino e teste."
)

A divisão agrupada conseguiu evitar a sobreposição dos grupos cadastrais e manteve a distribuição das classes praticamente igual à original. Isso atende ao objetivo dessa correção.

In [ ]:
print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

print("\nDistribuição no treino:")
print(y_train.value_counts(normalize=True))

print("\nDistribuição no teste:")
print(y_test.value_counts(normalize=True))

**Pré-processamento**

Temos dois tipos de variável.

**Numéricas**

Vamos:

preencher valores ausentes pela mediana;

padronizar.

**Categóricas**

Vamos:

preencher valores ausentes pela categoria mais frequente;

transformar categorias em variáveis numéricas com One-Hot Encoding.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

**Primeiro modelo: Regressão Logística**

In [ ]:
class_weight="balanced"

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Preprocessor criado com sucesso!")

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42,
                class_weight="balanced",
                C=0.5
            )
        )
    ]
)

logistic_model.fit(
    X_train,
    y_train
)

**Random Forest**

*Esse foi o modelo que apresentou o melhor desempenho geral no nosso experimento.*

In [ ]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=400,
                min_samples_leaf=5,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

*Treino*

In [ ]:
rf_model.fit(
    X_train,
    y_train
)

**Extra Trees**

*Nosso terceiro modelo*

In [ ]:
extra_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            ExtraTreesClassifier(
                n_estimators=400,
                min_samples_leaf=3,
                max_features="sqrt",
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

*Treino*

In [ ]:
extra_model.fit(
    X_train,
    y_train
)

**Função para Avaliação**

In [ ]:
def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            y_prob
        ),
        "PR-AUC": average_precision_score(
            y_test,
            y_prob
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            y_test,
            y_pred
        )
    }

    return metrics

**Avaliando os três modelos**

In [ ]:
results = []

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "Extra Trees": extra_model
}

for name, model in models.items():

    metrics = evaluate_model(
        model,
        X_test,
        y_test
    )

    metrics["Model"] = name

    results.append(metrics)

results_df = pd.DataFrame(results)

results_df = results_df[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC",
        "Balanced Accuracy"
    ]
]

display(results_df.round(6))

Os valores estão arredondados para três casas decimais.

Precision: a Regressão Logística tem precisão de aproximadamente 11,8%. Ou seja, entre os casos classificados como positivos, essa é a proporção de verdadeiros positivos.

F1: a Regressão Logística apresenta o maior F1, mas o resultado ainda é baixo.

ROC-AUC: os três modelos estão muito próximos de 0,5, indicando pouca capacidade de distinguir as classes pelo ranking das probabilidades.

PR-AUC: os resultados ficam próximos da prevalência da classe positiva, de aproximadamente 11,8%, indicando que o ganho em relação a uma referência simples é limitado.

Accuracy: o Random Forest tem a maior acurácia, mas seu recall de apenas 3,75% mostra que deixa passar a grande maioria dos inadimplentes.

Foi realizada uma análise de duplicidade na base cadastral, desconsiderando a coluna identificadora (ID) e comparando as demais variáveis. A investigação identificou 74.780 grupos de cadastros com atributos idênticos associados a IDs diferentes na base original.

Para reduzir o risco de vazamento de dados entre os conjuntos de treinamento e teste, foi implementada uma divisão estratificada e agrupada com o método StratifiedGroupKFold. Os grupos cadastrais foram mantidos integralmente em um único conjunto, sem sobreposição entre treino e teste.

A nova divisão contemplou 29.077 registros para treinamento e 7.380 para teste. A proporção da classe positiva (TARGET = 1) permaneceu próxima de 11,8% nos dois conjuntos. Após a alteração, os modelos foram treinados e avaliados novamente.

Os resultados indicaram limitações na capacidade discriminativa dos modelos, com valores de ROC-AUC próximos de 0,5. A Regressão Logística apresentou o maior recall e o maior F1 entre os modelos avaliados, enquanto o Random Forest apresentou maior acurácia, mas baixo recall para a classe positiva. Esses resultados reforçam a necessidade de avaliar múltiplas métricas e investigar estratégias adicionais de modelagem e definição do limiar de classificação.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

for nome, modelo in models.items():
    y_pred = modelo.predict(X_test)

    print("\n" + "=" * 50)
    print(f"Modelo: {nome}")
    print("=" * 50)

    print("\nMatriz de confusão:")
    print(confusion_matrix(y_test, y_pred))

    print("\nRelatório de classificação:")
    print(
        classification_report(
            y_test,
            y_pred,
            zero_division=0,
            digits=3
        )
    )

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

for nome, modelo in models.items():
    y_pred = modelo.predict(X_test)
    matriz = confusion_matrix(y_test, y_pred)

    print(f"\nModelo: {nome}")
    print("Matriz de confusão:")
    print(matriz)
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
    print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
    print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.3f}")
    print(f"F1:        {f1_score(y_test, y_pred, zero_division=0):.3f}")

**Matriz de confusão**

*para analisar Regressão Logística*

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Previsões da Regressão Logística no conjunto de teste
y_pred_log = logistic_model.predict(X_test)

# Matriz de confusão
cm_log = confusion_matrix(
    y_test,
    y_pred_log,
    labels=[0, 1]
)

# Gráfico
fig, ax = plt.subplots(figsize=(7, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_log,
    display_labels=["Bom pagador (0)", "Mau pagador (1)"]
)

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title("Matriz de Confusão — Regressão Logística")
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe real")

plt.tight_layout()
plt.show()

print("Matriz de confusão — Regressão Logística:")
print(cm_log)

A matriz de confusão permite avaliar o desempenho da Regressão Logística na classificação dos clientes entre bons e maus pagadores. Considerando a classe 1 como indicativa de mau pagador, o modelo identificou corretamente 370 dos 879 casos dessa classe, correspondendo a um recall de aproximadamente 42,1%. Por outro lado, classificou incorretamente 509 maus pagadores como bons pagadores.

Além disso, 2.774 clientes classificados como bons pagadores pelo modelo pertenciam, na realidade, à classe de maus pagadores? Não: esses 2.774 casos correspondem a bons pagadores classificados incorretamente como maus pagadores. Esse resultado evidencia a existência de erros nas duas classes e reforça a necessidade de avaliar o equilíbrio entre a identificação de potenciais inadimplentes e a redução de classificações incorretas.

**Matriz de confusão**

*para analisar o Random Forest*

In [ ]:
# Previsões da Random Forest no mesmo conjunto de teste
y_pred_rf = rf_model.predict(X_test)

# Matriz de confusão
cm_rf = confusion_matrix(
    y_test,
    y_pred_rf,
    labels=[0, 1]
)

# Gráfico
fig, ax = plt.subplots(figsize=(7, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=["Bom pagador (0)", "Mau pagador (1)"]
)

disp.plot(
    ax=ax,
    cmap="Greens",
    values_format="d",
    colorbar=False
)

ax.set_title("Matriz de Confusão — Random Forest")
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe real")

plt.tight_layout()
plt.show()

print("Matriz de confusão — Random Forest:")
print(cm_rf)

A matriz de confusão da Random Forest evidencia uma elevada capacidade de classificação dos clientes pertencentes à classe de bons pagadores, com 6322 acertos. Entretanto, o modelo identificou corretamente apenas 33 dos 879 maus pagadores presentes no conjunto de teste, correspondendo a um recall de aproximadamente 3,8% para a classe minoritária.

Além disso, 846 maus pagadores foram classificados incorretamente como bons pagadores, indicando uma limitação importante na identificação de clientes potencialmente inadimplentes. Embora o modelo apresente elevada acurácia global, esse indicador isoladamente não representa adequadamente seu desempenho, uma vez que a base de dados apresenta desequilíbrio entre as classes. Dessa forma, os resultados demonstram a necessidade de considerar métricas como recall, precisão, F1 e PR-AUC na avaliação da capacidade de identificação do risco de crédito.

** Importante **

A Regressão Logística foi melhor na identificação da classe de risco, enquanto a Random Forest privilegiou a classificação dos bons pagadores. 

**Curva ROC**

*A Curva ROC é uma ferramenta gráfica utilizada para avaliar o desempenho de um classificador binário, como os seus modelos. Ela plota a Taxa de Verdadeiros Positivos (TPR) contra a Taxa de Falsos Positivos (FPR) em vários limiares de classificação. Essencialmente, mostra a capacidade do modelo de distinguir entre as classes positivas (maus pagadores) e negativas (bons pagadores).*

In [ ]:
plt.figure(figsize=(8,6))

for name, model in models.items():

    y_prob = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(
        y_test,
        y_prob
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{name} — AUC={auc:.3f}"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC")
plt.legend()
plt.show()

O gráfico avalia o desempenho de três modelos na identificação de maus pagadores através da Curva ROC e da métrica AUC (Area Under the Curve). O Random Forest obtém o melhor resultado, com um AUC de 0,780, seguido de perto pelo Extra Trees, com 0,770. Ambos os modelos demonstram uma capacidade sólida para diferenciar clientes de bom e de mau risco. Em contraste, a Regressão Logística apresenta um desempenho fraco, com um AUC de 0,549, posicionando-se muito próxima de uma escolha aleatória (0,500). Assim, recomenda-se adotar o Random Forest como modelo oficial de decisão e descontinuar a Regressão Logística.   

**Curva Precision-Recall**

In [ ]:
plt.figure(figsize=(8,6))

for name, model in models.items():

    y_prob = model.predict_proba(X_test)[:, 1]

    precision, recall, _ = precision_recall_curve(
        y_test,
        y_prob
    )

    ap = average_precision_score(
        y_test,
        y_prob
    )

    plt.plot(
        recall,
        precision,
        label=f"{name} — AP={ap:.3f}"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall")
plt.legend()
plt.show()

O gráfico avalia a relação entre a precisão e a sensibilidade (recall) na identificação de maus pagadores através da Curva Precision-Recall e da métrica de Precisão Média (AP). O Random Forest apresenta o melhor desempenho, com uma precisão média de 0,376, seguido de perto pelo Extra Trees, com 0,368, demonstrando uma capacidade superior para equilibrar a captura de clientes de risco com a redução de falsos alarmes em cenários de dados desbalanceados. Em contrapartida, a Regressão Logística regista um resultado ineficaz, com um AP de apenas 0,142, confirmando a sua limitação para detetar a classe minoritária de risco. Conclui-se que o Random Forest é o modelo mais adequado para a operação, recomendando-se o ajuste fino do seu limiar de decisão (threshold) para otimizar o compromisso entre perdas por inadimplência e fricção na concessão de crédito.

**Variáveis mais influenciam a previsão**

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importancia_df = pd.DataFrame({
    "Variavel": X_test.columns,
    "Importancia_media": perm.importances_mean,
    "Desvio_padrao": perm.importances_std
}).sort_values(
    by="Importancia_media",
    ascending=False
)

display(importancia_df.round(3))

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

importance_df.head(15)

A importância das variáveis foi avaliada por meio da técnica de permutação (Permutation Importance), utilizando a métrica Average Precision. O procedimento consiste em embaralhar os valores de cada variável e observar a variação no desempenho do modelo.

Os resultados indicaram que OCCUPATION_TYPE, EMPLOYED_YEARS e NAME_EDUCATION_TYPE apresentaram as maiores importâncias médias para o Random Forest. Esses resultados sugerem que as informações relacionadas à ocupação profissional, ao tempo de emprego e à escolaridade contribuíram relativamente mais para o desempenho do modelo na avaliação realizada.

Por outro lado, a variável INCOME_PER_FAMILY apresentou importância média negativa, indicando que seu embaralhamento não prejudicou o desempenho médio observado. Esse resultado pode sugerir contribuição limitada da variável na configuração avaliada.

Entretanto, a importância das variáveis deve ser interpretada considerando o desempenho global do modelo. Como os indicadores de discriminação apresentaram valores baixos, não é possível concluir que essas variáveis sejam suficientes para distinguir adequadamente bons e maus pagadores. Os resultados devem ser considerados exploratórios, sem implicar relações causais entre as características dos solicitantes e o risco de inadimplência.

**Gráfico das principais variáveis**

In [ ]:
top_features = importance_df.head(10)

plt.figure(figsize=(10,6))

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title(
    "Principais Variáveis — Permutation Importance"
)

plt.xlabel(
    "Redução média no desempenho"
)

plt.ylabel("Variável")

plt.show()

O gráfico de Importância por Permutação destaca que a idade do cliente  e a posse de imóvel próprio são os fatores de maior impacto na previsão do risco de crédito. Logo em seguida, sobressaem-se os indicadores de estabilidade profissional e financeira, como o tipo de ocupação, a categoria de rendimento  e o tempo de emprego. Atributos como nível de escolaridade, renda logarítmica, genero, rendimento familiar por pessoa e estado civil apresentam um peso secundário e complementar na decisão. Estrategicamente, o modelo demonstra que a estabilidade patrimonial e profissional é mais determinante para avaliar o risco do que o valor absoluto da renda, devendo estas variáveis orientar prioritariamente as políticas de aprovação de crédito.

# ============================================
# RESULTADOS PROCESSADOS
# ============================================

In [ ]:
from pathlib import Path

# Pasta onde serão salvos os resultados processados
pasta_processed = Path("../data/processed")

# Cria a pasta caso ela não exista
pasta_processed.mkdir(parents=True, exist_ok=True)

# Salvar métricas dos modelos
arquivo_metricas = pasta_processed / "metricas_modelos.csv"
results_df.to_csv(arquivo_metricas, index=False)

# Salvar importância das variáveis
arquivo_importancia = pasta_processed / "importancia_variaveis.csv"
importance_df.to_csv(arquivo_importancia, index=False)

print("Arquivos salvos com sucesso!")
print(f"Métricas: {arquivo_metricas.resolve()}")
print(f"Importância das variáveis: {arquivo_importancia.resolve()}")


In [ ]:
from pathlib import Path

predictions = pd.DataFrame({
    "y_real": y_test.values,
    "y_pred": y_pred_rf,
    "probabilidade_mau_pagador": rf_model.predict_proba(X_test)[:, 1]
})

# Pasta de destino
pasta_processed = Path("../data/processed")
pasta_processed.mkdir(parents=True, exist_ok=True)

# Salvar diretamente em data/processed
arquivo_predicoes = pasta_processed / "predicoes_random_forest.csv"

predictions.to_csv(
    arquivo_predicoes,
    index=False
)

print("Predições salvas em:")
print(arquivo_predicoes.resolve())

1. A base de aplicação fornece as características do cliente.

2. A base de crédito fornece o comportamento histórico.

3. Criamos uma variável-alvo a partir do histórico.

4. Consideramos como mau pagador quem apresentou pelo menos um STATUS de 1 a 5.

5. Cruzamos o comportamento histórico com os dados cadastrais.

6. Fizemos limpeza e engenharia de atributos.

7. Dividimos os dados em treino e teste.

8. Criamos um pipeline de pré-processamento.

9. Treinamos três algoritmos diferentes.

10. Comparamos os modelos utilizando métricas adequadas ao desbalanceamento.

11. Random Forest apresentou o melhor desempenho geral.

12. Extra Trees teve maior capacidade de recuperar maus pagadores, mas com mais falsos positivos.

13. Finalmente, usamos permutation importance para entender quais variáveis tinham maior importância preditiva.